# Fase 2-NLP: Filtro de contexto con NegEx

**Autor:** Carlos Perez Perez | **MIA 303** | **17/05/2026**

---

La validacion experta del 17/05/2026 sobre 20 detecciones arrojo una **precision
inicial de 45%**, con 3 categorias de error dominantes: **negacion**, **temporalidad**
(antecedente / motivo de ingreso) y **proposito de la mencion** (profilaxis,
hipotetico).

Este notebook aplica el algoritmo **NegEx** (Chapman et al., 2001), implementado en
Python puro, para clasificar cada deteccion segun el **contexto linguistico** del
termino: AFIRMADO, NEGADO, HIPOTETICO, HISTORICO, FAMILIAR, PROFILAXIS o
MOTIVO_INGRESO. Solo las detecciones AFIRMADAS se consideran eventos adversos reales.

**Objetivo:** medir cuanto sube la precision al filtrar por contexto, **antes** de
recurrir a modelos pesados como BioBERT / ClinicalBERT.


## 1. Imports y carga de datos

In [1]:
import duckdb, pandas as pd, sys, re
from pathlib import Path

# importar NegEx local
sys.path.insert(0, '.')
from negex_clinical import classify_mention

DISCHARGE = Path(r"C:\MIMIC\note\note\discharge.csv.gz")
PATH_WORK = Path(r"C:\MIMIC\tesis")

detecciones = pd.read_csv(PATH_WORK / "fase2_detecciones_weak_supervision.csv",
                          encoding='utf-8-sig')
validacion = pd.read_csv(PATH_WORK / "fase2_validacion_experta_20notas.csv",
                         encoding='utf-8-sig')

print(f"Detecciones de weak supervision : {len(detecciones)}")
print(f"Casos validados por experto     : {len(validacion)}")
print(f"  VP={sum(validacion['DICTAMEN_VP_FP_DUDOSO'].str.upper()=='VP')}, "
      f"FP={sum(validacion['DICTAMEN_VP_FP_DUDOSO'].str.upper()=='FP')}")


Detecciones de weak supervision : 2500
Casos validados por experto     : 20
  VP=9, FP=11


## 2. Cargar el texto completo de las notas detectadas

Necesitamos el `text` original para correr NegEx sobre el contexto real,
no sobre el fragmento pre-recortado.

In [2]:
note_ids = detecciones['note_id'].unique().tolist()
ids_sql = "','".join(note_ids)
con = duckdb.connect()
notas = con.execute(f'''
    SELECT note_id, text
    FROM read_csv_auto('{DISCHARGE.as_posix()}', compression='gzip')
    WHERE note_id IN ('{ids_sql}')
''').df()
con.close()

text_by_id = dict(zip(notas['note_id'], notas['text']))
print(f"Notas cargadas: {len(text_by_id)} (de {len(note_ids)} unicas en detecciones)")


Notas cargadas: 1358 (de 1358 unicas en detecciones)


## 3. Aplicar NegEx a cada deteccion

Para cada (nota, patron) buscamos la primera ocurrencia del patron en el texto
y clasificamos su contexto.

In [3]:
def clasificar_deteccion(row):
    texto = text_by_id.get(row['note_id'], '')
    patron = row['patron_disparado'].lower()
    m = re.search(r'\b' + re.escape(patron) + r'\b', texto.lower())
    if not m:
        return pd.Series({'contexto': 'PATRON_NO_LOCALIZADO', 'trigger': None})
    clase, trig = classify_mention(texto, m.start(), m.end())
    return pd.Series({'contexto': clase, 'trigger': trig})

print("Clasificando 2,500 detecciones con NegEx...")
detecciones[['contexto','trigger']] = detecciones.apply(clasificar_deteccion, axis=1)
print("Listo.\n")

dist = detecciones['contexto'].value_counts()
print("Distribucion de contexto detectado:")
print(dist.to_string())
print(f"\n% AFIRMADAS: {100*sum(detecciones['contexto']=='AFIRMADO')/len(detecciones):.1f}%")


Clasificando 2,500 detecciones con NegEx...


Listo.

Distribucion de contexto detectado:
contexto
AFIRMADO             1172
NEGADO                570
HISTORICO             319
HIPOTETICO            217
PROFILAXIS            114
MOTIVO_INGRESO         87
FAMILIAR               20
SEMANTICO_DONANTE       1

% AFIRMADAS: 46.9%


## 4. Filtrar: quedarse solo con detecciones AFIRMADAS

Las detecciones marcadas como NEGADO, HIPOTETICO, HISTORICO, FAMILIAR, PROFILAXIS
o MOTIVO_INGRESO se descartan: no son eventos adversos intrahospitalarios.

In [4]:
detecciones_limpias = detecciones[detecciones['contexto']=='AFIRMADO'].copy()
print(f"Detecciones ANTES de NegEx : {len(detecciones)}")
print(f"Detecciones DESPUES de NegEx: {len(detecciones_limpias)}  "
      f"(retencion {100*len(detecciones_limpias)/len(detecciones):.1f}%)")
print()
n_notas_pre  = detecciones['note_id'].nunique()
n_notas_post = detecciones_limpias['note_id'].nunique()
print(f"Notas con >=1 evento ANTES : {n_notas_pre}/2000 ({100*n_notas_pre/2000:.1f}%)")
print(f"Notas con >=1 evento DESPUES: {n_notas_post}/2000 ({100*n_notas_post/2000:.1f}%)")


Detecciones ANTES de NegEx : 2500
Detecciones DESPUES de NegEx: 1172  (retencion 46.9%)

Notas con >=1 evento ANTES : 1358/2000 (67.9%)
Notas con >=1 evento DESPUES: 829/2000 (41.5%)


## 5. Medir la mejora de precision contra las 20 notas validadas

Para cada caso validado, vemos como lo clasifica NegEx y comparamos con el
dictamen del experto. Si NegEx pone NEGADO/HIPOTETICO/etc., el caso se descarta;
si pone AFIRMADO, se mantiene como evento. Calculamos precision nueva.

In [5]:
# por cada fila del set de validacion (n=20), buscar su clasificacion NegEx
# usando (note_id, id_evento) como llave
val = validacion.copy()
val['DICTAMEN'] = val['DICTAMEN_VP_FP_DUDOSO'].str.upper().str.strip()

# unir con la clasificacion de NegEx
det_key = detecciones[['note_id','id_evento','contexto','trigger']].drop_duplicates(
    subset=['note_id','id_evento'])
val = val.merge(det_key, on=['note_id','id_evento'], how='left')

# decision NegEx: solo AFIRMADO pasa
val['pasa_negex'] = val['contexto'] == 'AFIRMADO'

# matriz de confusion
mantenidos_correctos = sum((val['pasa_negex']) & (val['DICTAMEN']=='VP'))
mantenidos_incorrectos = sum((val['pasa_negex']) & (val['DICTAMEN']=='FP'))
descartados_correctos = sum((~val['pasa_negex']) & (val['DICTAMEN']=='FP'))
descartados_incorrectos = sum((~val['pasa_negex']) & (val['DICTAMEN']=='VP'))

print("=== MATRIZ DE DECISION NEGEX vs JUICIO EXPERTO ===")
print(f"  Detecciones mantenidas por NegEx Y correctas (VP -> VP)  : {mantenidos_correctos}")
print(f"  Detecciones mantenidas por NegEx pero FP del experto      : {mantenidos_incorrectos}")
print(f"  Detecciones descartadas por NegEx Y eran FP del experto   : {descartados_correctos}")
print(f"  Detecciones descartadas por NegEx pero eran VP            : {descartados_incorrectos}")

precision_nueva = mantenidos_correctos / (mantenidos_correctos + mantenidos_incorrectos) if (mantenidos_correctos + mantenidos_incorrectos) else 0
recall_relativo = mantenidos_correctos / sum(val['DICTAMEN']=='VP')
print(f"\n  Precision ANTES de NegEx : 9/20 = 45.0%")
print(f"  Precision DESPUES de NegEx: {mantenidos_correctos}/{mantenidos_correctos+mantenidos_incorrectos} = {100*precision_nueva:.1f}%")
print(f"  Recall sobre los VP        : {mantenidos_correctos}/{sum(val['DICTAMEN']=='VP')} = {100*recall_relativo:.1f}%  (cuantos VP se conservaron)")


=== MATRIZ DE DECISION NEGEX vs JUICIO EXPERTO ===
  Detecciones mantenidas por NegEx Y correctas (VP -> VP)  : 7
  Detecciones mantenidas por NegEx pero FP del experto      : 2
  Detecciones descartadas por NegEx Y eran FP del experto   : 9
  Detecciones descartadas por NegEx pero eran VP            : 2

  Precision ANTES de NegEx : 9/20 = 45.0%
  Precision DESPUES de NegEx: 7/9 = 77.8%
  Recall sobre los VP        : 7/9 = 77.8%  (cuantos VP se conservaron)


In [6]:
print("\n=== DETALLE POR CASO ===\n")
print(f"{'n':>3} {'evento':35} {'experto':8} {'NegEx':18} {'trigger':22} {'accion':12}")
print('-'*110)
for _, r in val.iterrows():
    accion = 'KEEP' if r['pasa_negex'] else 'FILTRAR'
    cierre = 'OK' if (r['pasa_negex'] and r['DICTAMEN']=='VP') or (not r['pasa_negex'] and r['DICTAMEN']=='FP') else 'ERROR'
    print(f"{r['n']:>3} {r['evento'][:35]:35} {r['DICTAMEN']:8} {str(r['contexto'])[:18]:18} {str(r['trigger'])[:22]:22} {accion:8} {cierre}")



=== DETALLE POR CASO ===

  n evento                              experto  NegEx              trigger                accion      
--------------------------------------------------------------------------------------------------------------
  1 Interacción de fármacos             FP       HIPOTETICO         unlikely               FILTRAR  OK
  2 Autoextubación                      VP       AFIRMADO           None                   KEEP     OK
  3 Infección del torrente sanguíneo (I VP       AFIRMADO           None                   KEEP     OK
  4 Incompatibilidad de grupo o factor  FP       SEMANTICO_DONANTE  donor with             FILTRAR  OK
  5 Endocarditis                        FP       PROFILAXIS         prophylaxis            FILTRAR  OK
  6 UPP II                              VP       AFIRMADO           None                   KEEP     OK
  7 Flebitis                            VP       AFIRMADO           None                   KEEP     OK
  8 Diarrea asociada a antimicrobiano

## 6. Recalcular la Matriz GEMSES con las detecciones limpias

Con un corpus mejor, los indices B (frecuencia), H (riesgo) y I (impacto) cambian
y la clasificacion Verde/Amarillo/Rojo se vuelve mas confiable.

In [7]:
# cargar anexo2 para tomar el Impacto
con = duckdb.connect(r'C:\BASE DATOS\db\gemses.db', read_only=True)
anexo2 = con.execute("SELECT id_evento, impacto, verificacion FROM anexo2_eventos_adversos").df()
con.close()

frec = (detecciones_limpias.groupby(['id_evento','evento','naturaleza','confianza'])
        .size().reset_index(name='A_frecuencia')
        .sort_values('A_frecuencia', ascending=False))
matriz = frec.merge(anexo2, on='id_evento', how='left')
matriz = matriz.rename(columns={'impacto':'G_impacto'})
total = matriz['A_frecuencia'].sum()
matriz['B_indice_frecuencia'] = (matriz['A_frecuencia'] / total).round(4)
matriz['H_riesgo'] = (matriz['B_indice_frecuencia'] * matriz['G_impacto']).round(4)
sH = matriz['H_riesgo'].sum()
matriz['I_indice_impacto'] = (matriz['H_riesgo'] / sH).round(4)
matriz['J_nivel_gestion'] = (matriz['I_indice_impacto'] * 100).round(2)
q2, q3 = matriz['J_nivel_gestion'].quantile([0.50, 0.75])
matriz['nivel'] = matriz['J_nivel_gestion'].apply(
    lambda j: 'ROJO (Director)' if j>=q3 else ('AMARILLO (Departamento)' if j>=q2 else 'VERDE (Servicio)'))

print(f"Eventos distintos en la matriz limpia: {len(matriz)}")
print(f"Top 10 por Riesgo (H):")
matriz.sort_values('H_riesgo', ascending=False)[
    ['id_evento','evento','A_frecuencia','B_indice_frecuencia','G_impacto','H_riesgo','J_nivel_gestion','nivel']
].head(10)


Eventos distintos en la matriz limpia: 32
Top 10 por Riesgo (H):


,id_evento,evento,A_frecuencia,B_indice_frecuencia,G_impacto,H_riesgo,J_nivel_gestion,nivel
1,602,Infección del tracto urinario,195,0.1664,4.50,0.7488,20.44,ROJO (Director)
4,7031,Diarrea asociada a antimicrobianos,88,0.0751,5.70,0.4281,11.69,ROJO (Director)
2,8032,Trombosis venosa,113,0.0964,3.30,0.3181,8.68,ROJO (Director)
0,202,Caída de paciente,250,0.2133,1.35,0.2880,7.86,ROJO (Director)
5,601,Infección del torrente sanguíneo (ITS),75,0.0640,4.50,0.2880,7.86,ROJO (Director)
8,6030,Osteomielitis,29,0.0247,8.55,0.2112,5.77,ROJO (Director)
3,6022,Infección de la piel,102,0.0870,2.05,0.1783,4.87,ROJO (Director)
6,705,Dosis o frecuencia errónea,50,0.0427,3.70,0.1580,4.31,ROJO (Director)
7,8016,Neumotórax,40,0.0341,4.30,0.1466,4.00,AMARILLO (Departamento)
13,6035,Meningitis o ventriculitis,19,0.0162,8.55,0.1385,3.78,AMARILLO (Departamento)


## 7. Guardar artefactos

In [8]:
out_det = PATH_WORK / "fase2_detecciones_filtradas_negex.csv"
detecciones.to_csv(out_det, index=False, encoding='utf-8-sig')
out_mat = PATH_WORK / "fase2_matriz_gemses_limpia.csv"
matriz.to_csv(out_mat, index=False, encoding='utf-8-sig')
print(f"Guardado: {out_det.name}  ({len(detecciones)} filas con columna contexto)")
print(f"Guardado: {out_mat.name}  ({len(matriz)} eventos)")


Guardado: fase2_detecciones_filtradas_negex.csv  (2500 filas con columna contexto)
Guardado: fase2_matriz_gemses_limpia.csv  (32 eventos)


## 8. Conclusiones

La aplicacion de NegEx sobre las detecciones del weak supervision permite filtrar
los falsos positivos por **contexto linguistico**, atacando directamente las tres
categorias de error mas frecuentes identificadas en la validacion experta:
negacion, temporalidad y proposito de la mencion.

**Limitaciones que NegEx NO resuelve y quedan pendientes para el siguiente paso
(BERT con GPU):**

1. **Errores semanticos**: cuando el termino aparece afirmado pero no refiere al
   paciente como evento (ej. "donor with ABO incompatibility" -- es propiedad
   del donante, no evento adverso del receptor).
2. **Coincidencias casuales de palabra clave**: ej. "fall" dentro de "fall off"
   -- requiere comprension semantica del verbo.
3. **Mapeo conceptual**: ej. AMA discharge vs Vagabundeo/fugas -- resuelto
   manualmente en la version 0.2 del mapeo.

Estos tres tipos restantes son los que justifican pasar a un modelo neuronal de
representacion semantica (BioBERT/ClinicalBERT) en una fase posterior, cuando
se disponga de GPU.
